In [ ]:
import torch
import cv2
import imageio.v3 as imageio
import torch.nn as nn
import torchvision.transforms.functional as F
from PIL import Image
from albumentations.augmentations.geometric.functional import resize
from mmm.api.M3Model import M3Model, M3_MODELS, DEFAULT_MODEL, TC_NOPANDA
from mmm.interactive import tasks  # interactive imports take a while
# We load the model from an archive
# which allows us to load only those modules that are compatible with the current environment.
model = M3Model(M3_MODELS[DEFAULT_MODEL], device_identifier="cuda:0")
model.get_device(), model.get_task_keys()[:5], model.get_sharedblock_keys()

# You can load the pathology specific encoder by using TC_NOPANDA in MMM_MODELS
# tissue_concepts = M3_MODELS(MMM_MODELS[TC_NOPANDA], device_identifier="cuda:0")

In [ ]:
# UMedPT-image link: https://owncloud.fraunhofer.de/index.php/s/n6gycdah9SaxOdD/download 
link = r"https://owncloud.fraunhofer.de/index.php/s/n6gycdah9SaxOdD/download"

# TC-image link: https://owncloud.fraunhofer.de/index.php/s/P5PAZ1gGUmYWNDy/download
# link = r"https://owncloud.fraunhofer.de/index.php/s/P5PAZ1gGUmYWNDy/download"

raw_image = imageio.imread(link)
Image.fromarray(raw_image)  # Display in Jupyter

In [ ]:
# The input should be divisible by 32 and similar to the pre-training setting.
# For ImageNet, we trained with a static size of 256x256
input_image = resize(raw_image, target_shape=(256, 256) ,interpolation=cv2.INTER_LINEAR)
# Inputs need to be between 0 and 1
input_image = F.to_tensor(input_image)

In [ ]:
with torch.inference_mode():
    # The model expects batches
    feature_pyramid = model["encoder"](input_image.unsqueeze(0).to(model.device))
    print([feature_map.shape for feature_map in feature_pyramid])
    hidden_vector = nn.Flatten(1)(model["squeezer"](feature_pyramid)[1])
    print(hidden_vector.shape)
    print(some_values := hidden_vector[0, :5].tolist())


In addition, we can perform inference with one of the pretraining tasks. The ImageNet classifier should classify the image of a teddy bear correctly as "teddy, teddy bear"

In [ ]:
mtl_task: tasks.ClassificationTask = model['imgnetclf']
# TC
# mtl_task: tasks.ClassificationTask = model['kather100kclf']
classnames = mtl_task.class_names.copy()

# You can use the representation for classification. 
scores = nn.Softmax(dim=1)(mtl_task.task_modules["classification_head"](hidden_vector).detach().cpu())
# Print the top classes
print([(classnames[i], scores[0, i].item()) for i in torch.argsort(scores, descending=True)[0, :5]])

In [ ]:
assert (highest_class := classnames[torch.argmax(scores).item()]) == "teddy, teddy bear"